## Imports and settings

In [83]:
import pandas as pd
import numpy as np
import pulp
import time
from pathlib import Path

DATA_FILE = Path("../data/cleaned_supply_chain.xlsx")

# Test 2, 3, 4, or 5 days
PLANNING_DAYS = 6

# Assumed delay penalty per order per day
DELAY_PENALTY_PER_ORDER_DAY = 100.0

TIME_LIMIT_SECONDS = 300

# CRF = customer-arranged freight, modeled at zero
# transportation cost to the company.
CRF_ZERO_FREIGHT = True

assert PLANNING_DAYS >= 1
assert DELAY_PENALTY_PER_ORDER_DAY >= 0

print("Planning days:", PLANNING_DAYS)
print("Delay penalty:", DELAY_PENALTY_PER_ORDER_DAY)
print("Dataset:", DATA_FILE.resolve())

Planning days: 6
Delay penalty: 100.0
Dataset: G:\MSC Academic\SEM Y1S2\IT5082 - Optimization Methods\op_assignment\OM---Assignment-MS26914596-MS26902852\data\cleaned_supply_chain.xlsx


## Load all seven Excel sheets

In [84]:
if not DATA_FILE.exists():
    raise FileNotFoundError(f'Update DATA_FILE to your cleaned workbook: {DATA_FILE}')

sheet_names = ['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities',
               'ProductsPerPlant', 'VmiCustomers', 'PlantPorts']
raw = pd.read_excel(DATA_FILE, sheet_name=sheet_names)
for name, df in raw.items():
    df.columns = df.columns.astype(str).str.strip()
    print(f'{name:20s}: {len(df):,} rows')

orders = raw['OrderList'].reset_index(drop=True).copy()
freight_rates = raw['FreightRates'].copy()
wh_costs = raw['WhCosts'].copy()
wh_capacities = raw['WhCapacities'].copy()
products_per_plant = raw['ProductsPerPlant'].copy()
vmi_customers = raw['VmiCustomers'].copy()
plant_ports = raw['PlantPorts'].copy()

OrderList           : 6,438 rows
FreightRates        : 1,540 rows
WhCosts             : 19 rows
WhCapacities        : 19 rows
ProductsPerPlant    : 2,036 rows
VmiCustomers        : 14 rows
PlantPorts          : 22 rows


## Validate input columns and numeric fields

In [85]:
required = {
    'OrderList': ['Order ID', 'Product ID', 'Customer', 'Service Level', 'Weight', 'Unit quantity', 'Destination Port'],
    'FreightRates': ['Carrier', 'orig_port_cd', 'dest_port_cd', 'svc_cd', 'minm_wgh_qty', 'max_wgh_qty', 'minimum cost', 'rate'],
    'WhCosts': ['WH', 'Cost/unit'],
    'WhCapacities': ['Plant ID', 'Daily Capacity'],
    'ProductsPerPlant': ['Plant Code', 'Product ID'],
    'VmiCustomers': ['Customers', 'Plant Code'],
    'PlantPorts': ['Plant Code', 'Port'],
}
for name, cols in required.items():
    missing = sorted(set(cols) - set(raw[name].columns))
    if missing:
        raise ValueError(f'{name} missing columns: {missing}')

numeric_fields = [
    (orders, ['Weight', 'Unit quantity']),
    (freight_rates, ['minm_wgh_qty', 'max_wgh_qty', 'minimum cost', 'rate']),
    (wh_costs, ['Cost/unit']),
    (wh_capacities, ['Daily Capacity']),
]
for df, cols in numeric_fields:
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='raise')
        if df[col].isna().any() or (df[col] < 0).any():
            raise ValueError(f'Invalid negative/missing values in {col}')

if (freight_rates['minm_wgh_qty'] > freight_rates['max_wgh_qty']).any():
    raise ValueError('Freight weight band min exceeds max')
if wh_capacities['Plant ID'].duplicated().any():
    raise ValueError('Duplicate plant capacity rows')
if wh_costs['WH'].duplicated().any():
    raise ValueError('Duplicate warehouse cost rows')
if orders[['Product ID', 'Customer', 'Service Level', 'Destination Port']].isna().any().any():
    raise ValueError('Missing required order attributes')
print('Input validation passed')

Input validation passed


## Build lookup dictionaries

In [86]:
print("Building lookup dictionaries...")

# Warehouse cost per unit
warehouse_cost = dict(
    zip(wh_costs["WH"], wh_costs["Cost/unit"])
)

# Daily plant capacity
capacity = dict(
    zip(
        wh_capacities["Plant ID"],
        wh_capacities["Daily Capacity"]
    )
)

all_plants = list(capacity.keys())

# Plant -> supported products
plant_products = (
    products_per_plant
    .groupby("Plant Code")["Product ID"]
    .apply(set)
    .to_dict()
)

# Plant -> available origin ports
plant_port_map = (
    plant_ports
    .groupby("Plant Code")["Port"]
    .apply(set)
    .to_dict()
)

# CORRECTED:
# VMI plant -> permitted customers
vmi_plant_map = (
    vmi_customers
    .groupby("Plant Code")["Customers"]
    .apply(set)
    .to_dict()
)

# Validate warehouse costs
missing_costs = set(all_plants) - set(warehouse_cost)

if missing_costs:
    raise ValueError(
        f"Missing warehouse costs: {missing_costs}"
    )

print("Total orders:", len(orders))
print("Total plants:", len(all_plants))
print("VMI-restricted plants:", len(vmi_plant_map))
print("Daily capacity:", sum(capacity.values()))
print(
    "Total planning capacity:",
    PLANNING_DAYS * sum(capacity.values())
)

Building lookup dictionaries...
Total orders: 6438
Total plants: 19
VMI-restricted plants: 4
Daily capacity: 5791
Total planning capacity: 34746


## Build and validate freight-rate lookup

**Important:** CRF orders are customer-arranged freight: they are charged zero freight and need no FreightRates row (the sheet contains no CRF rows).

In [87]:
print("Building freight-rate lookup...")

freight_lookup = {}

for _, row in freight_rates.iterrows():

    key = (
        row["Carrier"],
        row["orig_port_cd"],
        row["dest_port_cd"],
        row["svc_cd"]
    )

    freight_lookup.setdefault(key, []).append(
        (
            float(row["minm_wgh_qty"]),
            float(row["max_wgh_qty"]),
            float(row["minimum cost"]),
            float(row["rate"])
        )
    )


def get_freight_cost(
    carrier,
    origin_port,
    dest_port,
    service_level,
    weight
):
    # CRF: customer pays freight.
    # No FreightRates row is required.
    if service_level == "CRF" and CRF_ZERO_FREIGHT:
        return 0.0

    key = (
        carrier,
        origin_port,
        dest_port,
        service_level
    )

    bands = freight_lookup.get(key, [])

    matching_costs = []

    for min_w, max_w, minimum_cost, rate in bands:

        if min_w <= weight <= max_w:
            cost = max(
                minimum_cost,
                rate * weight
            )
            matching_costs.append(cost)

    if not matching_costs:
        return None

    return min(matching_costs)


print("Freight routes:", len(freight_lookup))
print(
    "CRF rate rows:",
    (freight_rates["svc_cd"] == "CRF").sum()
)
print(
    "CRF orders:",
    (orders["Service Level"] == "CRF").sum()
)

Building freight-rate lookup...
Freight routes: 44
CRF rate rows: 0
CRF orders: 695


## Generate feasible (order, plant, carrier) choices

Preserves product compatibility, VMI, plant-port connection, and freight eligibility. Keeps the **cheapest eligible origin port** for each order–plant–carrier combination (assuming no separate port capacity constraint).

In [88]:
print("Generating feasible assignments...")

all_carriers = (
    freight_rates["Carrier"]
    .dropna()
    .unique()
    .tolist()
)

feasible = {}
best_origin_port = {}
cost_parts = {}
skipped_orders = []

for i, row in orders.iterrows():

    product_id = row["Product ID"]
    customer = row["Customer"]
    service_level = row["Service Level"]
    weight = float(row["Weight"])
    quantity = float(row["Unit quantity"])
    dest_port = row["Destination Port"]

    found = False

    for plant in all_plants:

        # Rule 1: Product compatibility
        if product_id not in plant_products.get(
            plant, set()
        ):
            continue

        # Rule 2: Correct VMI restriction
        if (
            plant in vmi_plant_map
            and customer not in vmi_plant_map[plant]
        ):
            continue

        # Rule 3: Plant must have an origin port
        ports = plant_port_map.get(plant, set())

        if not ports:
            continue

        # Warehouse handling cost
        wh = float(warehouse_cost[plant]) * quantity

        for origin_port in ports:

            carriers = [
                "CRF (customer)"] if service_level == "CRF" else all_carriers
            for carrier in carriers:

                # Rule 4: Freight calculation
                fr = get_freight_cost(
                    carrier,
                    origin_port,
                    dest_port,
                    service_level,
                    weight
                )

                if fr is None:
                    continue

                key = (i, plant, carrier)
                total_cost = wh + fr

                # Keep cheapest origin port
                if (
                    key not in feasible
                    or total_cost < feasible[key]
                ):
                    feasible[key] = total_cost
                    best_origin_port[key] = origin_port
                    cost_parts[key] = (wh, fr)

                found = True

    if not found:
        skipped_orders.append(i)


valid_order_indices = sorted({
    key[0] for key in feasible
})

print("Original orders:", len(orders))
print("Eligible orders:", len(valid_order_indices))
print("Skipped orders:", len(skipped_orders))
print("Feasible combinations:", len(feasible))

print(
    "Eligible CRF orders:",
    sum(
        orders.at[i, "Service Level"] == "CRF"
        for i in valid_order_indices
    )
)

if skipped_orders:
    display(
        orders.loc[
            skipped_orders,
            [
                "Order ID",
                "Product ID",
                "Customer",
                "Service Level"
            ]
        ].head(20)
    )

if not valid_order_indices:
    raise ValueError("No feasible assignments found")

total_horizon_capacity = (
    PLANNING_DAYS * sum(capacity.values())
)

print("Planning capacity:", total_horizon_capacity)

if len(valid_order_indices) > total_horizon_capacity:
    print("WARNING: Insufficient total capacity")
else:
    print("Aggregate capacity check passed")

Generating feasible assignments...
Original orders: 6438
Eligible orders: 6438
Skipped orders: 0
Feasible combinations: 8633
Eligible CRF orders: 695
Planning capacity: 34746
Aggregate capacity check passed


## Define the multi-day MILP

`x[(i,p,c,t)]` is binary. `t=0` denotes Day 1. Each order consumes **one capacity slot** on the assigned day. No overflow variable is created.

In [89]:
model = pulp.LpProblem('MultiDay_SupplyChain_MILP', pulp.LpMinimize)
days = range(PLANNING_DAYS)

x = {}
by_order = {i: [] for i in valid_order_indices}
by_plant_day = {(p, t): [] for p in all_plants for t in days}

for j, ((i, p, c), base_cost) in enumerate(feasible.items()):
    for t in days:
        key = (i, p, c, t)
        x[key] = pulp.LpVariable(f'x_{j}_day{t+1}', cat='Binary')
        by_order[i].append(key)
        by_plant_day[(p, t)].append(key)

model += pulp.lpSum(
    (feasible[(i, p, c)] + DELAY_PENALTY_PER_ORDER_DAY * t) * var
    for (i, p, c, t), var in x.items()
), 'Logistics_Plus_Delay_Cost'

for i, keys in by_order.items():
    model += pulp.lpSum(x[k] for k in keys) == 1, f'Assign_{i}'

for j, p in enumerate(all_plants):
    for t in days:
        model += (pulp.lpSum(x[k] for k in by_plant_day[(p, t)]) <= float(capacity[p]),
                  f'Capacity_plant{j}_day{t+1}')

print('Binary variables:', len(x))
print('Assignment constraints:', len(valid_order_indices))
print('Capacity constraints:', len(all_plants) * PLANNING_DAYS)

Binary variables: 51798
Assignment constraints: 6438
Capacity constraints: 114


##  Solve using an exact MILP solver

HiGHS is preferred; CBC is a fallback. The **algorithm is exact**, but a time-limited run must not be called optimal unless the solver certifies it.

In [90]:
print("Solving MILP...")

available = pulp.listSolvers(onlyAvailable=True)

if "HiGHS" in available:
    solver = pulp.getSolver(
        "HiGHS",
        msg=True,
        timeLimit=TIME_LIMIT_SECONDS
    )

elif "PULP_CBC_CMD" in available:
    solver = pulp.PULP_CBC_CMD(
        msg=True,
        timeLimit=TIME_LIMIT_SECONDS
    )

else:
    raise RuntimeError("No MILP solver available")

start = time.perf_counter()

status_code = model.solve(solver)

solve_seconds = time.perf_counter() - start

status = pulp.LpStatus[status_code]
solution_status = pulp.LpSolution.get(
    model.sol_status, "Unknown"
)

print("Planning horizon:", PLANNING_DAYS)
print("Solver status:", status)
print("Solution status:", solution_status)
print(f"Runtime: {solve_seconds:.2f} seconds")

if (
    status == "Optimal"
    and model.sol_status == pulp.LpSolutionOptimal
):
    print(
        f"Optimal assignment found within "
        f"{PLANNING_DAYS} days."
    )
    print(
        "Objective value:",
        pulp.value(model.objective)
    )

elif status == "Infeasible":
    print(
        f"The {PLANNING_DAYS}-day model "
        "is infeasible."
    )

else:
    print(
        "Optimality not certified. "
        "Review the solver log."
    )

Solving MILP...
Planning horizon: 6
Solver status: Optimal
Solution status: Optimal Solution Found
Runtime: 4.31 seconds
Optimal assignment found within 6 days.
Objective value: 11161522.383234061


## Extract the chosen assignments

In [91]:
if status != 'Optimal':
    raise RuntimeError('No certified optimal solution; skip extraction. Inspect Cell 8 status.')

rows = []
for (i, p, c, t), var in x.items():
    if var.varValue is not None and var.varValue > 0.5:
        base_key = (i, p, c)
        wh, fr = cost_parts[base_key]
        rows.append({
            'Order_Index': i,
            'Order_ID': orders.at[i, 'Order ID'],
            'Product_ID': orders.at[i, 'Product ID'],
            'Customer': orders.at[i, 'Customer'],
            'Plant': p,
            'Carrier': c,
            'Origin_Port': best_origin_port[base_key],
            'Processing_Day': t + 1,
            'Warehouse_Cost': wh,
            'Freight_Cost': fr,
            'Delay_Cost': DELAY_PENALTY_PER_ORDER_DAY * t,
            'Total_Cost': wh + fr + DELAY_PENALTY_PER_ORDER_DAY * t
        })
results_df = pd.DataFrame(rows).sort_values(['Processing_Day', 'Plant', 'Order_Index']).reset_index(drop=True)
print('Assigned eligible orders:', len(results_df))
display(results_df.head(10))

Assigned eligible orders: 6438


,Order_Index,Order_ID,Product_ID,Customer,Plant,Carrier,Origin_Port,Processing_Day,Warehouse_Cost,Freight_Cost,Delay_Cost,Total_Cost
0,144,1.447338e+09,1699465,V5555555_30,PLANT03,CRF (customer),PORT04,1,1985.137256,0.0,0.0,1985.137256
1,145,1.447308e+09,1699465,V5555555_30,PLANT03,CRF (customer),PORT04,1,1051.046342,0.0,0.0,1051.046342
2,154,1.447185e+09,1699242,V5555555_30,PLANT03,CRF (customer),PORT04,1,441.429114,0.0,0.0,441.429114
3,157,1.447182e+09,1699799,V5555555_30,PLANT03,CRF (customer),PORT04,1,1457.285327,0.0,0.0,1457.285327
4,159,1.447397e+09,1699552,V5555555_30,PLANT03,CRF (customer),PORT04,1,1234.759513,0.0,0.0,1234.759513
5,161,1.447352e+09,1699552,V5555555_30,PLANT03,CRF (customer),PORT04,1,776.252837,0.0,0.0,776.252837
6,164,1.447178e+09,1699552,V5555555_30,PLANT03,CRF (customer),PORT04,1,3850.214074,0.0,0.0,3850.214074
7,166,1.447360e+09,1699552,V5555555_30,PLANT03,CRF (customer),PORT04,1,1227.514487,0.0,0.0,1227.514487
8,171,1.447206e+09,1693117,V5555555_30,PLANT03,CRF (customer),PORT04,1,172.328130,0.0,0.0,172.328130
9,172,1.447371e+09,1699172,V5555555_30,PLANT03,CRF (customer),PORT04,1,616.344753,0.0,0.0,616.344753


## Validate feasibility and reconcile the objective

In [92]:
assert len(results_df) == len(valid_order_indices), 'Not all eligible orders assigned'
assert results_df['Order_Index'].is_unique, 'Duplicate assignment of an order'
assert set(results_df['Order_Index']) == set(valid_order_indices), 'Order set mismatch'
assert results_df['Processing_Day'].between(1, PLANNING_DAYS).all()

for r in results_df.itertuples(index=False):
    assert (r.Order_Index, r.Plant, r.Carrier) in feasible
    assert r.Origin_Port == best_origin_port[(r.Order_Index, r.Plant, r.Carrier)]

usage = results_df.groupby(['Plant', 'Processing_Day']).size().to_dict()
for p in all_plants:
    for t in days:
        assigned = usage.get((p, t+1), 0)
        assert assigned <= float(capacity[p]) + 1e-6, f'Capacity violated for {p} day {t+1}'

cost_total = results_df[['Warehouse_Cost', 'Freight_Cost', 'Delay_Cost']].sum().sum()
assert np.isclose(cost_total, pulp.value(model.objective), rtol=1e-8, atol=0.01)
print('PASS: one assignment per eligible order')
print('PASS: daily plant capacity respected')
print('PASS: chosen routes feasible')
print('PASS: objective reconciled')
print(f'Orders outside the feasible set (not processed by this model): {len(skipped_orders)}')


print("\nChecking VMI restrictions...")

vmi_violations = []

for _, row in results_df.iterrows():

    plant = row["Plant"]
    customer = row["Customer"]

    if (
        plant in vmi_plant_map
        and customer not in vmi_plant_map[plant]
    ):
        vmi_violations.append(
            (row["Order_ID"], plant, customer)
        )

print("VMI violations:", len(vmi_violations))

assert len(vmi_violations) == 0, (
    "VMI restrictions violated!"
)


print("\nChecking CRF orders...")

original_crf_indices = set(
    orders.index[
        orders["Service Level"] == "CRF"
    ]
)

assigned_indices = set(
    results_df["Order_Index"]
)

missing_crf = (
    original_crf_indices - assigned_indices
)

print("Original CRF orders:", len(original_crf_indices))
print(
    "Assigned CRF orders:",
    len(original_crf_indices & assigned_indices)
)
print("Missing CRF orders:", len(missing_crf))

assert len(missing_crf) == 0, (
    "Some CRF orders were not assigned"
)

# Validate all original orders, not just eligible ones
print("\nChecking total assignment coverage...")

print("Original orders:", len(orders))
print("Assigned orders:", len(results_df))
print("Skipped orders:", len(skipped_orders))

assert len(skipped_orders) == 0, (
    "Some original orders are excluded. "
    "Investigate before reporting full coverage."
)

assert len(results_df) == len(orders), (
    "Not every original order was assigned."
)

print("\nAll additional validation checks passed!")

PASS: one assignment per eligible order
PASS: daily plant capacity respected
PASS: chosen routes feasible
PASS: objective reconciled
Orders outside the feasible set (not processed by this model): 0

Checking VMI restrictions...
VMI violations: 0

Checking CRF orders...
Original CRF orders: 695
Assigned CRF orders: 695
Missing CRF orders: 0

Checking total assignment coverage...
Original orders: 6438
Assigned orders: 6438
Skipped orders: 0

All additional validation checks passed!


## Daily plant utilization and cost summaries

In [93]:
# CELL 11 — Daily Plant Utilization and Cost Analysis

print("=" * 60)
print(f"MILP RESULTS — {PLANNING_DAYS}-DAY PLANNING HORIZON")
print("=" * 60)

# 1. Plant-wise daily utilization
summary_rows = []

for plant in all_plants:
    for t in days:

        processing_day = t + 1

        assigned = usage.get(
            (plant, processing_day), 0
        )

        cap = float(capacity[plant])

        summary_rows.append({
            "Plant": plant,
            "Day": processing_day,
            "Daily_Capacity": cap,
            "Assigned_Orders": assigned,
            "Remaining_Capacity": cap - assigned,
            "Utilization_Percent": (
                100 * assigned / cap
                if cap > 0 else np.nan
            )
        })

plant_day_df = pd.DataFrame(summary_rows)

# 2. Daily processing summary
daily_df = (
    results_df
    .groupby("Processing_Day")
    .agg(
        Orders=("Order_Index", "size"),
        Warehouse_Cost=("Warehouse_Cost", "sum"),
        Freight_Cost=("Freight_Cost", "sum"),
        Delay_Cost=("Delay_Cost", "sum"),
        Total_Cost=("Total_Cost", "sum")
    )
    .reset_index()
)

# 3. Carrier-wise summary
carrier_df = (
    results_df
    .groupby("Carrier")
    .agg(
        Orders=("Order_Index", "size"),
        Freight_Cost=("Freight_Cost", "sum")
    )
    .reset_index()
)

# 4. Cost breakdown
warehouse_total = results_df["Warehouse_Cost"].sum()
freight_total = results_df["Freight_Cost"].sum()
delay_total = results_df["Delay_Cost"].sum()
overall_total = results_df["Total_Cost"].sum()

print("\nCOST BREAKDOWN")
print(f"Warehouse Cost : {warehouse_total:,.2f}")
print(f"Freight Cost   : {freight_total:,.2f}")
print(f"Delay Cost     : {delay_total:,.2f}")
print(f"Total Cost     : {overall_total:,.2f}")

# 5. Order coverage
print("\nORDER COVERAGE")
print("Original orders :", len(orders))
print("Assigned orders :", len(results_df))
print("Skipped orders  :", len(skipped_orders))

# 6. Processing period
print("\nPROCESSING SUMMARY")
print(
    "Latest processing day:",
    results_df["Processing_Day"].max()
)

print(
    "Total plant capacity per day:",
    sum(capacity.values())
)

# 7. Verify strict capacity
violations = plant_day_df[
    plant_day_df["Assigned_Orders"]
    > plant_day_df["Daily_Capacity"]
]

assert violations.empty, (
    "Plant daily capacity violated!"
)

# 8. Verify objective cost
assert np.isclose(
    overall_total,
    pulp.value(model.objective),
    rtol=1e-7,
    atol=0.01
), "Objective value does not match cost breakdown"

print("\nDaily Processing Summary")
display(daily_df)

print("\nPlant Daily Utilization")
display(plant_day_df)

print("\nCarrier Summary")
display(carrier_df)

print("\nAll Cell 11 checks passed!")

MILP RESULTS — 6-DAY PLANNING HORIZON

COST BREAKDOWN
Warehouse Cost : 9,813,081.54
Freight Cost   : 67,940.85
Delay Cost     : 1,280,500.00
Total Cost     : 11,161,522.38

ORDER COVERAGE
Original orders : 6438
Assigned orders : 6438
Skipped orders  : 0

PROCESSING SUMMARY
Latest processing day: 6
Total plant capacity per day: 5791

Daily Processing Summary


,Processing_Day,Orders,Warehouse_Cost,Freight_Cost,Delay_Cost,Total_Cost
0,1,1829,2.545164e+06,60114.323595,0.0,2.605278e+06
1,2,1024,1.556214e+06,1748.658144,102400.0,1.660362e+06
2,3,1024,1.141991e+06,1699.999385,204800.0,1.348491e+06
3,4,1024,1.642900e+06,1828.611019,307200.0,1.951929e+06
4,5,1024,1.584252e+06,1630.713223,409600.0,1.995483e+06
5,6,513,1.342562e+06,918.540382,256500.0,1.599980e+06



Plant Daily Utilization


,Plant,Day,Daily_Capacity,Assigned_Orders,Remaining_Capacity,Utilization_Percent
0,PLANT15,1,11.0,0,11.0,0.0
1,PLANT15,2,11.0,0,11.0,0.0
2,PLANT15,3,11.0,0,11.0,0.0
3,PLANT15,4,11.0,0,11.0,0.0
4,PLANT15,5,11.0,0,11.0,0.0
...,...,...,...,...,...,...
109,PLANT04,2,554.0,0,554.0,0.0
110,PLANT04,3,554.0,0,554.0,0.0
111,PLANT04,4,554.0,0,554.0,0.0
112,PLANT04,5,554.0,0,554.0,0.0



Carrier Summary


,Carrier,Orders,Freight_Cost
0,CRF (customer),695,0.000000
1,V444_0,4964,65413.715894
2,V444_1,337,455.367600
3,V444_4,78,157.352422
4,V444_5,364,1914.409831



All Cell 11 checks passed!


## Save reproducible outputs

In [94]:
RESULTS_ROOT = Path("../results")


output_dir = (
    RESULTS_ROOT
    / f"MILP_{PLANNING_DAYS}days_penalty_{DELAY_PENALTY_PER_ORDER_DAY:g}"
)\

# Create the folder (and any missing parents) before writing
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / 'assignments.csv', index=False)
plant_day_df.to_csv(output_dir / 'plant_day_utilization.csv', index=False)
daily_df.to_csv(output_dir / 'daily_summary.csv', index=False)
carrier_df.to_csv(output_dir / 'carrier_summary.csv', index=False)
orders.loc[skipped_orders].to_csv(output_dir / 'unroutable_orders.csv', index=False)

metadata = pd.DataFrame([{
    'solver_status': status,
    'solve_seconds': solve_seconds,
    'planning_days': PLANNING_DAYS,
    'delay_penalty_per_order_day': DELAY_PENALTY_PER_ORDER_DAY,
    'original_orders': len(orders),
    'eligible_orders': len(valid_order_indices),
    'unroutable_orders': len(skipped_orders),
    'binary_variables': len(x),
    'objective': pulp.value(model.objective),
    'warehouse_cost': results_df['Warehouse_Cost'].sum(),
    'freight_cost': results_df['Freight_Cost'].sum(),
    'delay_cost': results_df['Delay_Cost'].sum()
}])
metadata.to_csv(output_dir / 'run_metadata.csv', index=False)
print('Saved outputs in:', output_dir.resolve())

Saved outputs in: G:\MSC Academic\SEM Y1S2\IT5082 - Optimization Methods\op_assignment\OM---Assignment-MS26914596-MS26902852\results\MILP_6days_penalty_100
